# I. NFL Local Batch — Design & Evidence Notebook

This notebook is **design documentation and execution evidence** for the NFL Local Batch data product.

It does **not** execute pipelines or create outputs.

It is used to:
- Define product boundary and scope
- Document source data characteristics using read-only inspection
- Establish grain, quality constraints, and metric eligibility
- Reference batch runs, outputs, and logs produced by scripts under `src/`

**Rule:** Explanation before execution. Evidence after execution.


## II. Product Definition

This section defines the product boundary before any analysis.
It documents who the product serves, what it supports, and what it does not support.

## Product Definition — Product Charter

### Product name
**NFL Historical Match Outcomes — Local Batch Metrics**

### Product type
- Offline, local batch data product
- Deterministic and reproducible
- Non-real-time
- Non-predictive

### Intended consumers
- Data analysts
- BI / reporting users
- Data engineers validating downstream pipelines
- Hiring reviewers (portfolio evaluation)

### Supported use cases
This product supports descriptive, historical analysis of NFL matches, including:
- Team-level outcome summaries across seasons
- Season-level aggregates (wins, losses, points for/against)
- Venue context indicators (home, away, neutral)
- Historical trend inspection (non-predictive)

All outputs are:
- Derived strictly from provided raw data
- Aggregated at clearly defined grains
- Suitable for retrospective reporting

### Explicit non-goals
This product does not support:
- Predictions, simulations, or forecasts
- Betting, odds optimisation, or financial modelling
- Player-level performance analysis
- Real-time or near-real-time use cases
- Causal inference or advanced statistical claims
- External data enrichment (weather, injuries, betting lines, etc.)

Any use case requiring the above is out of scope for v1.

### Source of truth & trust posture
- Raw data sourced from third-party public datasets
- No claim of official league accuracy
- Guarantees internal consistency; source accuracy is external and out of scope

### Published outputs (names only)
This product publishes the following batch metric tables:
- `team_outcomes`
- `season_summaries`
- `venue_neutral_counts`

Each output:
- Has an explicit grain
- Is governed by schema contracts
- Is produced via a logged batch run

### Product boundary statement
> This data product provides validated, contract-bound historical NFL match metrics for descriptive analysis only, executed as a reproducible local batch pipeline under a fixed framework.


## III. Source Data Inventory

The purpose of this section is to establish baseline facts about the raw input datasets
before any assumptions, transformations, or metric logic are applied.

This section documents:
- Which raw files are in scope
- Dataset sizes (row counts)
- Column inventories and data types
- Small, representative samples (for evidence only)

All checks are read-only.
No data is modified or written.


In [ ]:
#EXECUTION
import pandas as pd                                                        

teams_df = pd.read_csv("data/raw/nfl_teams.csv")
events_df = pd.read_csv("data/raw/spreadspoke_scores.csv")

print("Teams shape:", teams_df.shape)
print("Events shape:", events_df.shape)

print("\nTeams info:")
teams_df.info()

print("\nEvents info:")
events_df.info()

print("\nTeams head:")
display(teams_df.head(3))

print("\nEvents head:")
display(events_df.head(3))


Teams shape: (44, 8)
Events shape: (14358, 17)

Teams info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   team_name                44 non-null     object
 1   team_name_short          44 non-null     object
 2   team_id                  44 non-null     object
 3   team_id_pfr              44 non-null     object
 4   team_conference          44 non-null     object
 5   team_division            35 non-null     object
 6   team_conference_pre2002  44 non-null     object
 7   team_division_pre2002    42 non-null     object
dtypes: object(8)
memory usage: 2.9+ KB

Events info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14358 entries, 0 to 14357
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   schedule_date        14358 non-null  obj

,team_name,team_name_short,team_id,team_id_pfr,team_conference,team_division,team_conference_pre2002,team_division_pre2002
0,Arizona Cardinals,Cardinals,ARI,CRD,NFC,NFC West,NFC,NFC West
1,Atlanta Falcons,Falcons,ATL,ATL,NFC,NFC South,NFC,NFC West
2,Baltimore Colts,Colts,IND,CLT,AFC,NaN,AFC,AFC East



Events head:


,schedule_date,schedule_season,schedule_week,schedule_playoff,team_home,score_home,score_away,team_away,team_favorite_id,spread_favorite,over_under_line,stadium,stadium_neutral,weather_temperature,weather_wind_mph,weather_humidity,weather_detail
0,9/2/1966,1966,1,False,Miami Dolphins,14.0,23.0,Oakland Raiders,NaN,NaN,NaN,Orange Bowl,False,83.0,6.0,71.0,NaN
1,9/3/1966,1966,1,False,Houston Oilers,45.0,7.0,Denver Broncos,NaN,NaN,NaN,Rice Stadium,False,81.0,7.0,70.0,NaN
2,9/4/1966,1966,1,False,San Diego Chargers,27.0,7.0,Buffalo Bills,NaN,NaN,NaN,Balboa Stadium,False,70.0,7.0,82.0,NaN


## Source Data Inventory — Conclusions

- Two raw datasets are in scope:
  - `spreadspoke_scores.csv` — event-level fact dataset
  - `nfl_teams.csv` — team reference dataset
- Column inventories and data types were established via executed read-only inspection.
- No filtering, enrichment, or transformation was applied at this stage.
- These datasets form the sole inputs for all downstream batch metrics.


## IV. Data Grain & Quality Assessment

Before computing any metrics, we must establish:
- The dataset grain (what one row represents)
- Column completeness constraints (missingness)
- Whether exact duplicate rows exist

These checks provide auditable evidence for:
- Metric eligibility decisions
- Contract nullability and key constraints

All checks are read-only inspections.
No transformations or outputs are produced here.


In [5]:
# Columns 
print("Teams columns:")
display(pd.DataFrame({"column": teams_df.columns}))

print("\nEvents columns:")
display(pd.DataFrame({"column": events_df.columns}))

# Missingness (count + %)
def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    miss_count = df.isna().sum()
    miss_pct = (miss_count / len(df) * 100).round(2)
    out = (
        pd.DataFrame({"missing_count": miss_count, "missing_pct": miss_pct})
        .sort_values(["missing_count", "missing_pct"], ascending=False)
    )
    return out

teams_missing = missing_summary(teams_df)
events_missing = missing_summary(events_df)

print("\nTeams missingness summary:")
display(teams_missing)

print("\nEvents missingness summary:")
display(events_missing)

# Exact duplicate rows
teams_dup_rows = teams_df.duplicated().sum()
events_dup_rows = events_df.duplicated().sum()

print(f"\nExact duplicate rows | Teams: {teams_dup_rows} | Events: {events_dup_rows}")


Teams columns:


,column
0,team_name
1,team_name_short
2,team_id
3,team_id_pfr
4,team_conference
5,team_division
6,team_conference_pre2002
7,team_division_pre2002



Events columns:


,column
0,schedule_date
1,schedule_season
2,schedule_week
3,schedule_playoff
4,team_home
5,score_home
6,score_away
7,team_away
8,team_favorite_id
9,spread_favorite



Teams missingness summary:


,missing_count,missing_pct
team_division,9,20.45
team_division_pre2002,2,4.55
team_name,0,0.00
team_name_short,0,0.00
team_id,0,0.00
team_id_pfr,0,0.00
team_conference,0,0.00
team_conference_pre2002,0,0.00



Events missingness summary:


,missing_count,missing_pct
weather_detail,11048,76.95
weather_humidity,5674,39.52
over_under_line,2505,17.45
team_favorite_id,2495,17.38
spread_favorite,2495,17.38
weather_wind_mph,1549,10.79
weather_temperature,1532,10.67
score_home,16,0.11
score_away,16,0.11
schedule_date,0,0.00



Exact duplicate rows | Teams: 0 | Events: 0


## Data Grain & Quality Assessment — Conclusions

### Grain confirmation
- `spreadspoke_scores.csv`:
  - One row represents **one scheduled NFL game** between two teams.
- `nfl_teams.csv`:
  - One row represents **one NFL team** (historical or current).

### Duplicate assessment
- Exact duplicate rows:
  - Teams dataset: **0**
  - Events dataset: **0**
- Duplicate rows are not present and do not require remediation.

### Missingness assessment
- **Teams dataset**
  - Core identifiers (`team_id`, `team_name`, `team_name_short`) are fully populated.
  - Division fields contain partial historical gaps and are treated as optional attributes.

- **Events dataset**
  - Core game identifiers and outcome fields (`schedule_date`, `schedule_season`,
    `team_home`, `team_away`, `score_home`, `score_away`) are effectively complete.
  - Weather-related fields exhibit high missingness and are considered optional context only.
  - Betting-related fields (`spread_favorite`, `over_under_line`, `team_favorite_id`)
    have partial coverage and are not relied upon for required metrics.

### Quality implication
- Data quality is sufficient for **descriptive, historical match outcome metrics**.
- Metric computation will rely only on fields with near-complete coverage.
- Optional fields with high missingness are excluded from required contracts.


## V. Metric Eligibility Determination

This section determines which framework-defined metric categories are eligible
for computation based strictly on evidence established in prior sections.

Eligibility decisions are constrained by:
- Confirmed data grain
- Field completeness
- Transformation minimisation principles

---

### Eligibility Mapping

| Metric category (framework-defined) | Eligible | Evidence-based rationale |
|-----------------------------------|:-------:|--------------------------|
| Match counts and participation metrics | Yes | Event-level grain (one row per match) with complete season and team identifiers. |
| Win / loss outcome summaries | Yes | Home and away scores are present with near-complete coverage and consistent semantics. |
| Venue usage and neutrality summaries | Yes | Stadium and neutral venue indicators are fully populated. |
| Temporal distributions (seasonal, yearly) | Yes | Schedule date, season, and playoff indicators are complete and stable. |
| Dataset completeness and coverage metrics | Yes | Missingness has been assessed and scoped. |
| Team reference enrichment metrics | No | Team reference dataset does not add required attributes for v1 outputs and is not joined. |

---

### Explicit Exclusions
- No predictive, causal, or prescriptive metrics (framework non-goals).
- No enrichment-based metrics requiring team metadata.
- No metrics dependent on sparse betting or weather fields.

---

### Eligibility Conclusion
Based on evidence from Sections III and IV, all v1 metrics are computed
**exclusively from the event-level dataset**.
Reference datasets are excluded in v1 to keep the transformation surface minimal.


## VI. Contract References

This section documents the data contracts enforced for this data product.

Contracts exist only where a dataset actively participates in metric computation
or output publication.

---

### Active Contracts

#### `raw_events_contract.py`
**Dataset:** `spreadspoke_scores.csv`  
**Role:** Sole source dataset for all metrics.

**Guarantees:**
- Structural integrity and required fields
- One row represents one NFL game
- Exact duplicate rows are reported (not removed); deduplication is out of scope.
- No silent coercion or imputation

---

#### `metrics_contracts.py`
**Datasets:**
- `team_outcomes`
- `season_summaries`
- `venue_neutral_counts`

**Guarantees:**
- Output schema, types, and grain are explicit
- Metric completeness rules are enforced
- Outputs are deterministic and reproducible

---

### Reference-Only Datasets

#### `nfl_teams.csv`
- Inspected during exploration (Sections III–IV)
- No joins or enrichments performed
- No metrics depend on this dataset
- No contracts enforced

This dataset is retained for potential future enrichment in a subsequent
version (v1.1), not in the current scope.

---

### Contract Governance Statement
Contracts are scoped strictly to datasets that materially affect outputs.
Unused datasets are explicitly excluded to avoid false guarantees.


## VII. Batch Run Evidence

This section documents execution evidence for the local batch pipeline.
Outputs are staged publish (no partial outputs).
The pipeline is executed outside the notebook via the runner module.
The notebook records **what was executed and what was produced**, not the execution itself.

The purpose is to demonstrate:
- Successful end-to-end execution
- Contract enforcement (raw + outputs)
- Deterministic artifact publication
- Availability of audit logs for reproducibility

---

### Execution Record
```bash
python -m src.runner.run_local_batch


In [1]:
import json
from pathlib import Path
import pandas as pd

print("TEAM OUTCOMES (team_outcomes.parquet)---------")
team_outcomes = pd.read_parquet("outputs/metrics/team_outcomes.parquet")
display(team_outcomes.head())
print("Row count:", len(team_outcomes))

print("\n SEASON SUMMARIES (season_summaries.parquet)---------")
season_summaries = pd.read_parquet("outputs/metrics/season_summaries.parquet")
display(season_summaries.head())
print("Row count:", len(season_summaries))

print("\n NEUTRAL VENUE COUNTS (venue_neutral_counts.parquet)------------")
venue_neutral_counts = pd.read_parquet("outputs/metrics/venue_neutral_counts.parquet")
display(venue_neutral_counts.head())
print("Row count:", len(venue_neutral_counts))


print("\n RUN LOG (latest)------------")
log_files = sorted(Path("logs").glob("run_*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
latest_log = log_files[0]
print("Log file:", latest_log.name)

with open(latest_log) as f:
    run_log = json.load(f)

display(run_log)


TEAM OUTCOMES (team_outcomes.parquet)---------


,team,season,games_played,wins,losses,ties,points_for,points_against
0,Atlanta Falcons,1966,14,3,11,0,204,437
1,Baltimore Colts,1966,14,9,5,0,314,226
2,Boston Patriots,1966,7,4,2,1,158,146
3,Buffalo Bills,1966,15,9,5,1,365,286
4,Chicago Bears,1966,14,5,7,2,234,272


Row count: 1774

 SEASON SUMMARIES (season_summaries.parquet)---------


,season,games_total,playoff_games,regular_games
0,1966,171,3,168
1,1967,180,5,175
2,1968,188,6,182
3,1969,189,7,182
4,1970,189,7,182


Row count: 60

 NEUTRAL VENUE COUNTS (venue_neutral_counts.parquet)------------


,season,neutral_games
0,1966,1
1,1967,1
2,1968,2
3,1969,1
4,1970,1


Row count: 60

 RUN LOG (latest)------------
Log file: run_c547f0fcc34940d7b64c71c01c0ff19c.json


{'run_id': 'c547f0fcc34940d7b64c71c01c0ff19c',
 'started_at_utc': '2026-02-08T17:25:02.667914+00:00',
 'finished_at_utc': '2026-02-08T17:25:02.766368+00:00',
 'inputs': {'events': 'raw_data/spreadspoke_scores.csv'},
 'row_counts': {'raw_events': 14358,
  'events': 14358,
  'team_outcomes': 1774,
  'season_summaries': 60,
  'venue_neutral_counts': 60},
 'validations': {'raw_events_contract': {'result': 'PASS', 'severity': 'HARD'},
  'reconciliation_event_count_preserved': {'result': 'PASS',
   'severity': 'HARD'},
  'team_outcomes_contract': {'result': 'PASS', 'severity': 'HARD'},
  'season_summaries_contract': {'result': 'PASS', 'severity': 'HARD'},
  'venue_neutral_counts_contract': {'result': 'PASS', 'severity': 'HARD'}},
 'outputs': {'team_outcomes': 'outputs/metrics/team_outcomes.parquet',
  'season_summaries': 'outputs/metrics/season_summaries.parquet',
  'venue_neutral_counts': 'outputs/metrics/venue_neutral_counts.parquet'},
 'status': 'SUCCESS',
 'error': None}

### Conclusion

The local batch pipeline executed successfully and produced all expected
artifacts in a single deterministic run.

- The raw events dataset (`14358` rows) passed contract validation with no
  structural or duplication failures.
- Three metric tables were generated and validated:
  - `team_outcomes` — `1774` team-season records
  - `season_summaries` — `60` season-level records
  - `venue_neutral_counts` — `60` season-level records
- All output datasets conform to their declared schemas and passed metric-level
  validation rules.
- Outputs were published as Parquet files,
  indicating a complete and non-partial run.
- A run log (`run_c547f0fcc34940d7b64c71c01c0ff19c.json`) was emitted, capturing
  execution metadata, validation results, row counts, and output locations.

This evidence confirms that the local batch data product operates as designed,
with enforceable contracts, deterministic outputs, and auditable execution.


## VIII. Product Boundaries & Limitations

This local batch data product is intentionally scoped and constrained.

**Boundaries**
- Metrics are derived exclusively from the historical event-level dataset
  (`spreadspoke_scores.csv`).
- No enrichment from reference datasets (e.g., teams metadata) is performed in v1.
- Outputs are descriptive summaries only; no predictive, causal, or prescriptive
  logic is included.
- The pipeline operates in local batch mode and is not designed for real-time
  or near-real-time processing.

**Limitations**
- Games with missing scores are excluded from outcome-based metrics by design;
  no imputation is applied.
- Betting and weather fields are present in the raw data but are not used due to
  partial coverage and non-essential relevance to required metrics.
- Team identity normalization across historical eras is explicitly out of scope.
- Data quality guarantees are limited to structural and metric-level constraints;
  external source accuracy is not verified.

These boundaries keep transformations minimal, auditable, and deterministic
outputs aligned with the framework’s non-goals.


## IX. Reproducibility Notes

This data product is designed to be fully reproducible under the same inputs
and code version.

**Reproducibility guarantees**
- Raw input data is preserved unchanged under `data/raw/`.
- All transformations and metrics are deterministic.
- Contracts enforce schema and rule consistency before and after computation.
- Outputs are written in a stable, schema-aware format (Parquet).

**Execution requirements**
- Python environment with dependencies listed in `requirements.txt`
- Local execution via:
  ```bash
  python -m src.runner.run_local_batch
